# **CP462 Project: Crop Recommendation System**

# หัวข้อ : **ระบบแนะนำชนิดพืชปลูกที่เหมาะสมจากค่าดินและสภาพอากาศ (Crop Recommendation)**

**สมาชิก:**
<br>1) [นาย พลภัทร เพ็ชรสินธพ]
<br>2) [นาย อภินัทธ์ ยศวราเลิศ]
<br>3) [นาย ชัยสิทธิ์ ปลื้มสุทธิ์]
<br><br>
**Dataset:** Crop Recommendation Dataset <br>
**Source:** Kaggle - [atharvaingle/crop-recommendation-dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)



## **Import Packages**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยให้กราฟแสดงข้อความไทยได้ถูกต้อง (ไม่มี glyph missing warning)
!apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1

for font_path in fm.findSystemFonts():
    if "Waree" in font_path or "waree" in font_path:
        fm.fontManager.addfont(font_path)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# ต้องตั้งฟอนต์ "หลังจาก" sns.set_style เสมอ เพราะ seaborn style จะ reset font.family กลับเป็นค่า default
plt.rcParams["font.family"] = "Waree"
plt.rcParams["axes.unicode_minus"] = False

%matplotlib inline
print(plt.rcParams["font.family"])
import matplotlib.font_manager as fm
print([f.name for f in fm.fontManager.ttflist if 'aree' in f.name.lower()][:5])


['Waree']
['Waree', 'Waree', 'Waree', 'Waree', 'Waree']


## **Import Dataset**




In [ ]:
!pip install -q kagglehub
import kagglehub
dataset_path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
print("Dataset downloaded to:", dataset_path)

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
Dataset downloaded to: /kaggle/input/crop-recommendation-dataset


In [ ]:
import os

# หาไฟล์ .csv ภายใน path ที่ดาวน์โหลดมา แล้วอ่านเข้าเป็น DataFrame
csv_file = [f for f in os.listdir(dataset_path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(dataset_path, csv_file))

print("Shape ของข้อมูล:", df.shape)
df.head()

Shape ของข้อมูล: (2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


## **Explore Data Analysis**

In [ ]:
# ภาพรวมโครงสร้างข้อมูล (dtype, non-null count)
df.info()

# สถิติเชิงพรรณนาของข้อมูลตัวเลข (mean, std, min, max, quartiles)
df.describe()

# ตรวจสอบค่าที่หายไป (missing values)
print("จำนวนค่าที่หายไปในแต่ละคอลัมน์:")
print(df.isnull().sum())

# ตรวจสอบแถวข้อมูลที่ซ้ำกัน
print("\nจำนวนแถวข้อมูลที่ซ้ำกัน (duplicate rows):", df.duplicated().sum())

# จำนวนตัวอย่างต่อชนิดพืช (class distribution)
print("\nจำนวนตัวอย่างต่อชนิดพืช (label):")
print(df["label"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   object 
dtypes: float64(4), int64(3), object(1)
memory usage: 137.6+ KB
จำนวนค่าที่หายไปในแต่ละคอลัมน์:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

จำนวนแถวข้อมูลที่ซ้ำกัน (duplicate rows): 0

จำนวนตัวอย่างต่อชนิดพืช (label):
label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram  

In [ ]:
# GUI สำหรับเลือกดูกราฟ EDA แต่ละแบบผ่าน dropdown
import ipywidgets as widgets
from IPython.display import display, clear_output

features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

# ฟังก์ชันสำหรับกราฟ/สรุปข้อมูลแต่ละแบบ ---------------------------------

def show_overview():
    print("สรุปภาพรวมข้อมูล (Data Overview)")
    print(f"- จำนวนแถวทั้งหมด: {df.shape[0]:,} แถว")
    print(f"- จำนวนคอลัมน์: {df.shape[1]} คอลัมน์")
    print(f"- จำนวนชนิดพืช (label): {df['label'].nunique()} ชนิด")
    print(f"- ค่าที่หายไป (missing values): {df.isnull().sum().sum()} ค่า")
    print(f"- แถวข้อมูลที่ซ้ำกัน (duplicate rows): {df.duplicated().sum()} แถว")
    print()
    display(df[features].describe().T)

def show_distribution():
    fig, axes = plt.subplots(2, 4, figsize=(18, 7))
    for ax, col in zip(axes.flatten(), features):
        sns.histplot(df[col], kde=True, ax=ax, color="seagreen")
        ax.set_title(f"Distribution of {col}")
    axes.flatten()[-1].axis("off")
    plt.tight_layout()
    plt.show()

def show_correlation():
    plt.figure(figsize=(7, 5.5))
    sns.heatmap(df[features].corr(), annot=True, cmap="YlGnBu", fmt=".2f")
    plt.title("Correlation Heatmap ของค่าดินและสภาพอากาศ")
    plt.tight_layout()
    plt.show()

def show_class_balance():
    plt.figure(figsize=(12, 4.5))
    df["label"].value_counts().plot(kind="bar", color="darkorange")
    plt.title("จำนวนตัวอย่างต่อชนิดพืช (Class Distribution)")
    plt.xlabel("ชนิดพืช")
    plt.ylabel("จำนวนตัวอย่าง")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

def show_boxplot():
    plt.figure(figsize=(15, 4.5))
    sns.boxplot(x="label", y="N", data=df)
    plt.xticks(rotation=90)
    plt.title("ปริมาณไนโตรเจน (N) ที่เหมาะสมของแต่ละชนิดพืช")
    plt.tight_layout()
    plt.show()

def show_crop_profile():
    label_options = sorted(df["label"].unique())
    crop_dropdown = widgets.Dropdown(
        options=label_options,
        value=label_options[0],
        description="เลือกพืชที่จะปลูก:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="420px"),
    )
    crop_out = widgets.Output()

    feature_th = {
        "N": "ไนโตรเจน (N) mg/kg",
        "P": "ฟอสฟอรัส (P) mg/kg",
        "K": "โพแทสเซียม (K) mg/kg",
        "temperature": "อุณหภูมิ (C)",
        "humidity": "ความชื้นสัมพัทธ์ (%)",
        "ph": "ค่า pH ดิน",
        "rainfall": "ปริมาณน้ำฝน (mm)",
    }

    def render(crop):
        with crop_out:
            clear_output(wait=True)
            sub = df[df["label"] == crop]
            rows = []
            for f in features:
                rows.append({
                    "ปัจจัย": feature_th[f],
                    "ต่ำสุด": round(sub[f].min(), 1),
                    "เฉลี่ย": round(sub[f].mean(), 1),
                    "สูงสุด": round(sub[f].max(), 1),
                })
            profile = pd.DataFrame(rows).set_index("ปัจจัย")
            print("ช่วงค่าที่เหมาะสมสำหรับปลูก: " + crop + "  (จากตัวอย่าง " + str(len(sub)) + " แปลง)")
            print("")
            display(profile)

            fig, ax = plt.subplots(figsize=(9, 4))
            for i, f in enumerate(features):
                data_min, data_max = df[f].min(), df[f].max()
                span = data_max - data_min
                lo = (sub[f].min() - data_min) / span * 100
                hi = (sub[f].max() - data_min) / span * 100
                ax.barh(i, hi - lo, left=lo, height=0.5, color="#5E8A50")
            ax.set_yticks(range(len(features)))
            ax.set_yticklabels([feature_th[f] for f in features])
            ax.set_xlim(0, 100)
            ax.set_xlabel("ตำแหน่งเทียบกับช่วงค่าต่ำสุด-สูงสุดทั้งชุดข้อมูล (%)")
            ax.set_title("พืช '" + crop + "' ต้องการปัจจัยแต่ละตัวอยู่ระดับไหน (0%=ต่ำสุด, 100%=สูงสุด)")
            ax.invert_yaxis()
            plt.tight_layout()
            plt.show()

            print("อ่านง่ายๆ: แท่งอยู่ทางซ้าย = พืชนี้ต้องการค่านั้นต่ำ, แท่งอยู่ทางขวา = ต้องการค่านั้นสูง")
            print("ใช้ตารางด้านบนเทียบกับผลตรวจดินของแปลงตัวเองได้เลยว่าดินพร้อมปลูกพืชนี้หรือยัง")

    def on_crop_change(change):
        if change["name"] == "value":
            render(change["new"])

    crop_dropdown.observe(on_crop_change, names="value")
    display(widgets.VBox([crop_dropdown, crop_out]))
    render(crop_dropdown.value)

# GUI แบบ dropdown เลือกดูกราฟ ------------------------------------------

graph_options = {
    "📊 สรุปภาพรวมข้อมูล (Overview)": show_overview,
    "📈 การกระจายตัวของแต่ละตัวแปร (Distribution)": show_distribution,
    "🔥 ความสัมพันธ์ระหว่างตัวแปร (Correlation Heatmap)": show_correlation,
    "🌾 จำนวนตัวอย่างต่อชนิดพืช (Class Distribution)": show_class_balance,
    "📦 ไนโตรเจนของแต่ละชนิดพืช (Boxplot N by label)": show_boxplot,
    "🌱 คู่มือปลูก: ค่าที่เหมาะสมต่อพืชแต่ละชนิด (Crop Profile)": show_crop_profile,
}

dropdown = widgets.Dropdown(
    options=list(graph_options.keys()),
    value=list(graph_options.keys())[0],
    description="เลือกดูข้อมูล:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="480px"),
)
out = widgets.Output()

def on_change(change):
    if change["name"] == "value":
        with out:
            clear_output(wait=True)
            graph_options[change["new"]]()

dropdown.observe(on_change, names="value")

display(widgets.VBox([dropdown, out]))
with out:
    graph_options[dropdown.value]()


### **ตรวจ Outlier (IQR) ทุกคอลัมน์**
ใช้เกณฑ์ IQR: ค่าที่ต่ำกว่า Q1 − 1.5·IQR หรือสูงกว่า Q3 + 1.5·IQR ถือเป็น outlier
ตรวจทั้งแบบ **ทั้งชุดข้อมูล** และแบบ **แยกตามชนิดพืช** เพราะพืชแต่ละชนิดต้องการค่าต่างกันมาก

In [ ]:
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

def iqr_outlier_mask(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

# 1) outlier เมื่อมองทั้งชุดข้อมูล
overall = {f: int(iqr_outlier_mask(df[f]).sum()) for f in features}

# 2) outlier เมื่อมองภายในพืชชนิดเดียวกัน (groupby label)
within_crop = {
    f: int(df.groupby("label")[f].transform(iqr_outlier_mask).sum()) for f in features
}

outlier_table = pd.DataFrame({"outlier (ทั้งชุดข้อมูล)": overall,
                              "outlier (ภายในพืชชนิดเดียวกัน)": within_crop})
outlier_table

In [ ]:
# ดูว่า outlier ของ K และ P (ทั้งชุดข้อมูล) มาจากพืชชนิดไหน
for f in ["P", "K", "rainfall"]:
    mask = iqr_outlier_mask(df[f])
    print(f"{f}: outlier {mask.sum()} แถว มาจากพืช ->", df.loc[mask, "label"].value_counts().to_dict())

**ผลและการตัดสินใจ:** outlier ที่พบเมื่อมองทั้งชุดข้อมูลมาจาก **พืชเพียงไม่กี่ชนิดที่ต้องการค่านั้นสูงจริง**
(เช่น apple / grapes ต้องการ P และ K สูงมาก, rice ต้องการน้ำฝนมาก) ไม่ใช่ค่าที่บันทึกผิด
เมื่อมองภายในพืชชนิดเดียวกัน **ไม่พบ outlier เลย (0 ทุกคอลัมน์)**
→ **เราเก็บข้อมูลไว้ทั้งหมด ไม่ตัดออก** เพราะค่าเหล่านี้คือ "ลายเซ็น" ที่ใช้แยกพืชแต่ละชนิด ถ้าตัดทิ้ง โมเดลจะแยกพืชเหล่านั้นไม่ได้

### **ทดสอบสมมติฐาน: P กับ K สัมพันธ์กันจริงหรือไม่**
- H₀: ค่าสหสัมพันธ์ระหว่าง P กับ K = 0
- H₁: ค่าสหสัมพันธ์ระหว่าง P กับ K ≠ 0  (α = 0.05)

In [ ]:
from scipy import stats

r, p_value = stats.pearsonr(df["P"], df["K"])
rho, p_spear = stats.spearmanr(df["P"], df["K"])
print(f"Pearson  r   = {r:.3f}, p-value = {p_value:.2e}")
print(f"Spearman rho = {rho:.3f}, p-value = {p_spear:.2e}")
print("ปฏิเสธ H0 -> P กับ K สัมพันธ์กันอย่างมีนัยสำคัญ" if p_value < 0.05 else "ไม่ปฏิเสธ H0")

**ข้อสังเกต:** Pearson r ≈ 0.74 แต่ Spearman ต่ำกว่ามาก แปลว่าความสัมพันธ์ส่วนใหญ่ **ถูกดึงขึ้นโดยพืชกลุ่มเดียว (apple, grapes)** ที่ P และ K สูงทั้งคู่
ไม่ใช่ความสัมพันธ์เชิงเส้นที่เกิดทั่วทั้งชุดข้อมูล — และเป็น correlation ไม่ใช่ causation

## **Summary project progress**

**สรุปความคืบหน้า Part 1**
1. **ปัญหา:** เกษตรกรรายย่อยเลือกพืชจากความเคยชิน ไม่ได้อิงค่าดิน/อากาศจริง → ต้นทุนปุ๋ยสูงแต่ผลผลิตไม่คุ้ม
2. **ข้อมูล:** 2,200 แถว, 7 features ตัวเลข + 1 target (22 ชนิดพืช ชนิดละ 100 แถว) — ไม่มี missing, ไม่มี duplicate, คลาสสมดุล
3. **สิ่งที่พบจาก EDA:** N แยกกลุ่มพืชได้ชัด, P กับ K สัมพันธ์กันสูง (r = 0.74) ซึ่งมาจากพืชกลุ่ม apple/grapes เป็นหลัก, outlier ที่พบเป็นค่าจริงของพืชบางชนิดจึงเก็บไว้
4. **ก้าวต่อไป (Part 2):** แบ่ง train/test → Pipeline → เทียบ Baseline (DummyClassifier), Logistic Regression, Gaussian Naive Bayes, Decision Tree ด้วย Cross-validation + GridSearchCV → ประเมินบน test set

## **Data pre-processing**

### **ประเภทงาน (Type of learning / Task)**
- **Supervised Learning** — มีคำตอบ (label = ชนิดพืช) ให้โมเดลเรียนรู้
- **Multi-class Classification** — ทำนายพืช 1 ใน **22 ชนิด** (target เป็นหมวดหมู่ ไม่ใช่ตัวเลขต่อเนื่อง จึงไม่ใช่ regression)

### **การเลือก Feature**
ใช้ทั้ง 7 ตัว: `N, P, K, temperature, humidity, ph, rainfall`
- ทุกตัวเป็นค่าที่ **วัดได้ก่อนเริ่มปลูก** (ผลตรวจดิน + ข้อมูลอากาศ) → ใช้งานได้จริงตอนเกษตรกรตัดสินใจ
- แม้ P กับ K จะสัมพันธ์กัน (r = 0.74) แต่ยังเก็บไว้ทั้งคู่ เพราะ K ช่วยแยก apple กับ grapes ออกจากพืชอื่น
  - **Gaussian Naive Bayes** สมมติว่า features เป็นอิสระต่อกัน *ภายในแต่ละคลาส* — เมื่อดูภายในพืชชนิดเดียวกัน P กับ K แทบไม่สัมพันธ์กัน (r ≈ 0, ดู cell ด้านล่าง) ความสัมพันธ์ 0.74 มาจากความต่างระหว่างพืช จึงไม่ขัดกับสมมติฐานของ NB
  - **Decision Tree** ไม่เสียหายจาก multicollinearity และ **Logistic Regression** ใช้ regularization (ปรับด้วย `C`) ช่วยลดผลกระทบ

### **ขั้นตอนป้องกัน data leakage**
1. **แบ่ง train/test ก่อน** ทำ preprocessing ใดๆ (80/20, `stratify` ให้ทุกพืชมีสัดส่วนเท่ากันทั้งสองชุด)
2. ใส่ `StandardScaler` ไว้ **ใน `Pipeline`** → ตอนทำ cross-validation scaler จะถูก fit จาก fold ที่ใช้เทรนเท่านั้น ไม่เห็นข้อมูล validation/test
3. test set ถูกเก็บไว้ใช้ **ครั้งเดียวตอนท้าย** หลังเลือกโมเดลเสร็จแล้ว

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

X = df[features]
y_text = df["label"]

# แปลง label (ชื่อพืช) เป็นตัวเลข 0-21 — เก็บ encoder ไว้แปลงกลับเป็นชื่อพืช
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

# แบ่ง train/test ก่อนทำ preprocessing อื่นๆ ทั้งหมด
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("จำนวนตัวอย่างต่อพืชใน train (min-max):", np.bincount(y_train).min(), "-", np.bincount(y_train).max())
print("จำนวนตัวอย่างต่อพืชใน test  (min-max):", np.bincount(y_test).min(), "-", np.bincount(y_test).max())

In [ ]:
# ตรวจ target leakage / ข้อมูลรั่ว
# 1) มีแถวไหนใน test ที่ซ้ำกับ train ทุกค่าหรือไม่ (ถ้ามี = โมเดลเคยเห็นข้อสอบมาแล้ว)
overlap = pd.merge(X_train, X_test, how="inner", on=features)
print("แถวที่ซ้ำกันระหว่าง train กับ test:", len(overlap))

# 2) ไม่มี feature ไหนที่สร้างจาก label หรือเป็นข้อมูลหลังเก็บเกี่ยว
print("Features ที่ใช้:", features)
print("'label' อยู่ใน features หรือไม่:", "label" in features)

**ผลการตรวจ leakage:** ไม่มีแถวซ้ำระหว่าง train กับ test และไม่มี feature ใดที่สร้างมาจาก label หรือเป็นข้อมูลที่รู้หลังปลูก
→ ผลประเมินบน test set สะท้อนการใช้งานจริงกับแปลงที่โมเดลไม่เคยเห็น

## **Build Machine Learning Model**

### **โมเดลที่เลือกและเหตุผล** (โมเดลที่เรียนในห้องเรียน + baseline)
| โมเดล | เรียนใน | ประเภท | เหตุผลที่เลือก |
|---|---|---|---|
| **Baseline – DummyClassifier (uniform)** | Week 13 | baseline | สุ่มทายพืชแบบเท่าๆ กัน = "เดาโดยไม่ดู features" ใช้เป็นเส้นขั้นต่ำ |
| **Baseline – DummyClassifier (most_frequent)** | Week 13 | baseline | ทายพืชชนิดเดียวทุกครั้ง — ใช้แสดง *accuracy paradox* (accuracy 4.5% แต่ F1 เกือบ 0) |
| **Logistic Regression** | Week 12–13 | linear model | โมเดลเชิงเส้นที่ง่ายที่สุด ให้ความน่าจะเป็นผ่าน softmax ใช้วัดว่าข้อมูลแยกด้วยเส้นตรงได้แค่ไหน |
| **Gaussian Naive Bayes** | Week 12 (Lab) | probabilistic | เรียน mean/std ของแต่ละ feature ในพืชแต่ละชนิด แล้วใช้ Bayes' theorem — พืชแต่ละชนิดมี "ช่วงค่าดิน" ของตัวเองคล้ายรูประฆังคว่ำ ตรงกับสมมติฐาน Gaussian |
| **Decision Tree** | Week 14 | tree | ได้กฎแบบ if-else ที่อธิบายให้เกษตรกร/เจ้าหน้าที่เข้าใจได้ เช่น "ถ้า K > 150 → apple/grapes" และจับขอบเขตที่ไม่เป็นเส้นตรงได้ |

ทุกโมเดลอยู่ใน `Pipeline(StandardScaler → model)` เหมือนกัน และใช้ CV ชุดเดียวกัน (fold เดียวกันทุกโมเดล) เพื่อให้เทียบได้อย่างยุติธรรม
(scaler จำเป็นสำหรับ Logistic Regression ที่ใช้ regularization; สำหรับ Naive Bayes และ Decision Tree ไม่มีผลต่อผลลัพธ์แต่ใส่ไว้ให้ขั้นตอนเหมือนกันทุกโมเดล)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV

def make_pipeline(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])

BASELINES = ["Baseline (uniform)", "Baseline (most_frequent)"]

models = {
    "Baseline (uniform)":       make_pipeline(DummyClassifier(strategy="uniform", random_state=RANDOM_STATE)),
    "Baseline (most_frequent)": make_pipeline(DummyClassifier(strategy="most_frequent")),
    "Logistic Regression":      make_pipeline(LogisticRegression(max_iter=5000)),
    "Gaussian Naive Bayes":     make_pipeline(GaussianNB()),
    "Decision Tree":            make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
}

# Stratified 5-fold CV บน train set เท่านั้น
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "f1_macro": "f1_macro"}

# ตรวจสมมติฐานของ Naive Bayes: ภายในพืชชนิดเดียวกัน P กับ K สัมพันธ์กันแค่ไหน
within_pk = df.groupby("label")[["P", "K"]].apply(lambda g: g["P"].corr(g["K"]))
print(f"P–K correlation ทั้งชุดข้อมูล: r = {df['P'].corr(df['K']):.3f}")
print(f"P–K correlation ภายในพืชเดียวกัน: เฉลี่ย r = {within_pk.mean():.3f} (|r| สูงสุด = {within_pk.abs().max():.3f})")

### **ขั้นที่ 1: Cross-validation ด้วยค่า default (ก่อน tuning)**
รายงานเป็น **mean ± std** ของ 5 folds — std บอกว่าผลนิ่งแค่ไหนเมื่อข้อมูลเปลี่ยนชุด

In [ ]:
def cv_summary(pipe):
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    return {
        "CV Accuracy (mean)": res["test_accuracy"].mean(),
        "CV Accuracy (std)":  res["test_accuracy"].std(),
        "CV Macro-F1 (mean)": res["test_f1_macro"].mean(),
        "CV Macro-F1 (std)":  res["test_f1_macro"].std(),
    }

cv_default = pd.DataFrame({name: cv_summary(pipe) for name, pipe in models.items()}).T
cv_default.style.format("{:.4f}")

### **ขั้นที่ 2: Hyperparameter tuning ด้วย GridSearchCV**
ใช้ CV ชุดเดิม (5-fold บน train) และเลือกค่าที่ดีที่สุดจาก **Macro-F1**

| โมเดล | Hyperparameter ที่ปรับ | เหตุผล |
|---|---|---|
| Logistic Regression | `C` (0.01 – 100) | ความแรงของ regularization — C น้อย = บังคับให้เส้นแบ่งเรียบมาก (เสี่ยง underfit) |
| Gaussian Naive Bayes | `var_smoothing` (1e-9 – 1e-6) | เพิ่มค่าเล็กๆ ให้ variance ป้องกันการหารด้วยค่าที่เล็กเกินไป |
| Decision Tree | `max_depth`, `min_samples_leaf`, `criterion` | คุมความลึก/ขนาดใบ ป้องกันต้นไม้จำข้อมูล (overfit) |

In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "Gaussian Naive Bayes": {
        "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6],
    },
    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [5, 10, 15, None],
        "model__min_samples_leaf": [1, 2, 5],
    },
}

best_models = {name: models[name].fit(X_train, y_train) for name in BASELINES}
tuning_rows = []

for name, grid in param_grids.items():
    search = GridSearchCV(models[name], grid, cv=cv, scoring="f1_macro", n_jobs=-1)
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    i = search.best_index_
    tuning_rows.append({
        "Model": name,
        "Best params": str({k.replace("model__", ""): v for k, v in search.best_params_.items()}),
        "CV Macro-F1 (mean ± std)": f"{search.cv_results_['mean_test_score'][i]:.4f} ± {search.cv_results_['std_test_score'][i]:.4f}",
        "จำนวน combination ที่ลอง": len(search.cv_results_["params"]),
    })

pd.set_option("display.max_colwidth", None)  # แสดง best params แบบเต็ม ไม่ตัดข้อความ
pd.DataFrame(tuning_rows).set_index("Model")

In [ ]:
# CV ของโมเดลหลัง tuning (accuracy, macro-F1 และ train F1 เพื่ออ่าน bias–variance) เทียบกับ baseline
def cv_summary_with_train(pipe):
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    return {
        "CV Accuracy (mean)": res["test_accuracy"].mean(),
        "CV Macro-F1 (mean)": res["test_f1_macro"].mean(),
        "CV Macro-F1 (std)":  res["test_f1_macro"].std(),
        "Train Macro-F1":     res["train_f1_macro"].mean(),
    }

cv_tuned = pd.DataFrame({name: cv_summary_with_train(pipe) for name, pipe in best_models.items()}).T
display(cv_tuned.style.format("{:.4f}"))

# กราฟเปรียบเทียบ CV Macro-F1 (mean ± std) — แกน y เริ่มที่ 0 เพื่อให้เห็นระยะห่างจาก baseline ตามจริง
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#B0B0B0" if n in BASELINES else "#5E8A50" for n in cv_tuned.index]
ax.bar(cv_tuned.index, cv_tuned["CV Macro-F1 (mean)"], yerr=cv_tuned["CV Macro-F1 (std)"],
       capsize=6, color=colors)
for i, v in enumerate(cv_tuned["CV Macro-F1 (mean)"]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
ax.set_ylim(0, 1.1)
ax.set_ylabel("CV Macro-F1 (5-fold, train set)")
ax.set_title("เปรียบเทียบโมเดลหลัง tuning กับ Baseline (mean ± std)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## **Evaluate Model**
### **ตัวชี้วัดที่ใช้และเหตุผล**
- **Accuracy** — สัดส่วนที่แนะนำพืชถูก เข้าใจง่ายสำหรับผู้ใช้ และใช้ได้เพราะคลาสสมดุล (ทุกพืช 100 แถว)
- **Macro-F1** — ค่าเฉลี่ย F1 ของทุกพืชแบบให้น้ำหนักเท่ากัน → ระบบต้องแนะนำได้ดี **ทุกพืช** ไม่ใช่เก่งแค่บางชนิด
  (ถ้าแนะนำพืชผิดสำหรับเกษตรกรกลุ่มใดกลุ่มหนึ่ง = ต้นทุนเสียไปทั้งฤดู)
- **Confusion Matrix** — ดูว่าพืชคู่ไหนที่โมเดลยังสับสนกัน

**การเลือกโมเดล:** ตัดสินจาก **CV Macro-F1 บน train set** (ไม่ใช่จาก test set เพื่อไม่ให้ test รั่วเข้าไปในการตัดสินใจ)
แล้วใช้ test set เพียงครั้งเดียวเพื่อยืนยันผล

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

test_rows = []
for name, pipe in best_models.items():
    y_pred = pipe.predict(X_test)
    test_rows.append({
        "Model": name,
        "CV Macro-F1": cv_tuned.loc[name, "CV Macro-F1 (mean)"],
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Macro-Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Test Macro-Recall": recall_score(y_test, y_pred, average="macro"),
        "Test Macro-F1": f1_score(y_test, y_pred, average="macro"),
        "ทายผิด (แถว)": int((y_pred != y_test).sum()),
    })

results = pd.DataFrame(test_rows).set_index("Model")
baseline_acc = results.loc["Baseline (uniform)", "Test Accuracy"]
results["Accuracy ดีกว่า baseline สุ่มทาย (เท่า)"] = results["Test Accuracy"] / baseline_acc
results.style.format({c: "{:.4f}" for c in results.columns if c.startswith(("CV", "Test"))} |
                     {"Accuracy ดีกว่า baseline สุ่มทาย (เท่า)": "{:.1f}x"})

In [ ]:
# เลือกโมเดลที่ดีที่สุดจาก CV Macro-F1 (ไม่รวม baseline)
best_name = cv_tuned.drop(BASELINES)["CV Macro-F1 (mean)"].idxmax()
best_model = best_models[best_name]
print("โมเดลที่เลือก:", best_name)

y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=label_encoder.classes_, digits=3))

In [ ]:
# Confusion Matrix ของโมเดลที่เลือก
fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_best), display_labels=label_encoder.classes_
).plot(ax=ax, cmap="Greens", colorbar=False, xticks_rotation=90)
ax.set_title(f"Confusion Matrix — {best_name} (test set {len(y_test)} แถว)")
plt.tight_layout()
plt.show()

In [ ]:
# พืชคู่ไหนที่แต่ละโมเดลยังทายสลับกัน — ช่วยอธิบายจุดอ่อนของแต่ละโมเดล
confused = []
for name, pipe in best_models.items():
    if name in BASELINES:
        continue
    pred = pipe.predict(X_test)
    for t, p in zip(y_test[pred != y_test], pred[pred != y_test]):
        confused.append({"Model": name,
                         "พืชจริง": label_encoder.classes_[t],
                         "โมเดลทายเป็น": label_encoder.classes_[p]})

confused_df = pd.DataFrame(confused)
if confused_df.empty:
    print("ไม่มีโมเดลไหนทายผิดเลยบน test set")
else:
    display(confused_df.value_counts().rename("จำนวนครั้ง").reset_index())

### **Feature Importance — ปัจจัยไหนสำคัญต่อการเลือกพืช**
ใช้ **Permutation Importance** บน test set (สุ่มสลับค่าของ feature ทีละตัว แล้วดูว่า accuracy ตกลงเท่าไร)
ใช้ได้กับทุกโมเดล รวมถึง Naive Bayes ที่ไม่มี feature importance ในตัว

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_model, X_test, y_test, n_repeats=20,
                              random_state=RANDOM_STATE, scoring="accuracy")
imp = pd.Series(perm.importances_mean, index=features).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index, imp.values, xerr=perm.importances_std[np.argsort(perm.importances_mean)], color="#5E8A50")
ax.set_xlabel("Accuracy ที่ลดลงเมื่อสลับค่า feature นั้น")
ax.set_title(f"Permutation Importance — {best_name}")
plt.tight_layout()
plt.show()
imp.sort_values(ascending=False).round(4)

### **ทดลองใช้งานจริง: ใส่ค่าดิน/อากาศของแปลง → ได้พืชแนะนำ 3 อันดับ**
แนะนำแบบ top-3 พร้อมความมั่นใจ เพื่อให้เกษตรกรมีทางเลือก (เช่นเลือกพืชที่มีตลาดรับซื้อในพื้นที่)

In [ ]:
def recommend_crops(N, P, K, temperature, humidity, ph, rainfall, top_k=3, model=best_model):
    sample = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=features)
    proba = model.predict_proba(sample)[0]
    top = np.argsort(proba)[::-1][:top_k]
    return pd.DataFrame({
        "อันดับ": range(1, top_k + 1),
        "พืชที่แนะนำ": label_encoder.inverse_transform(top),
        "ความมั่นใจ (%)": (proba[top] * 100).round(1),
    }).set_index("อันดับ")

# ตัวอย่าง: แปลงที่ N สูง ฝนชุก ความชื้นสูง (ลักษณะคล้ายนาข้าว)
print("ตัวอย่างที่ 1: N=90, P=42, K=43, 21°C, ความชื้น 82%, pH 6.5, ฝน 203 mm")
display(recommend_crops(90, 42, 43, 20.9, 82.0, 6.5, 202.9))

# ตัวอย่าง: ดินที่ P และ K สูงมาก อากาศเย็น
print("ตัวอย่างที่ 2: N=20, P=130, K=200, 22°C, ความชื้น 92%, pH 6.0, ฝน 110 mm")
display(recommend_crops(20, 130, 200, 22.0, 92.0, 6.0, 110.0))

### **GUI: ระบบแนะนำพืชแบบกรอกค่าเอง**
เลื่อน slider ใส่ค่าผลตรวจดินและอากาศของแปลง แล้วกดปุ่ม **แนะนำพืช** → ได้พืช 3 อันดับพร้อมความมั่นใจ
(ใช้โมเดล Gaussian Naive Bayes ที่เลือกไว้ · ช่วงของ slider = ค่าต่ำสุด-สูงสุดในชุดข้อมูล)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

feature_th = {
    "N": "ไนโตรเจน N", "P": "ฟอสฟอรัส P", "K": "โพแทสเซียม K",
    "temperature": "อุณหภูมิ (°C)", "humidity": "ความชื้น (%)",
    "ph": "ค่า pH ดิน", "rainfall": "ปริมาณฝน (mm)",
}

# slider ของแต่ละ feature: ช่วง = min-max ของข้อมูล, ค่าเริ่มต้น = median
sliders = {}
for f in features:
    lo, hi, mid = float(df[f].min()), float(df[f].max()), float(df[f].median())
    is_int = f in ["N", "P", "K"]
    cls = widgets.IntSlider if is_int else widgets.FloatSlider
    kwargs = dict(min=int(lo), max=int(hi), value=int(mid), step=1) if is_int else \
             dict(min=round(lo, 1), max=round(hi, 1), value=round(mid, 1), step=0.1)
    sliders[f] = cls(description=feature_th[f], continuous_update=False,
                     style={"description_width": "150px"},
                     layout=widgets.Layout(width="520px"), **kwargs)

button = widgets.Button(description="🌱 แนะนำพืช", button_style="success",
                        layout=widgets.Layout(width="200px", height="40px"))
gui_out = widgets.Output()

def on_recommend(_):
    with gui_out:
        clear_output(wait=True)
        values = {f: s.value for f, s in sliders.items()}
        result = recommend_crops(**values, top_k=3)
        print("ผลแนะนำสำหรับแปลงนี้:")
        display(result)

        fig, ax = plt.subplots(figsize=(6, 2.4))
        ax.barh(result["พืชที่แนะนำ"][::-1], result["ความมั่นใจ (%)"][::-1], color="#5E8A50")
        ax.set_xlim(0, 100)
        ax.set_xlabel("ความมั่นใจ (%)")
        ax.set_title("พืชที่แนะนำ 3 อันดับ")
        plt.tight_layout()
        plt.show()

button.on_click(on_recommend)
display(widgets.VBox([*sliders.values(), button, gui_out]))
on_recommend(None)  # แสดงผลของค่าเริ่มต้นทันที

### **บันทึกโมเดล (Save Model)**
เซฟทั้ง Pipeline (StandardScaler + Gaussian Naive Bayes), LabelEncoder และรายชื่อ features ไว้ในไฟล์เดียวด้วย `joblib`
เพื่อนำไปใช้ต่อ (เช่น ทำแอป) ได้โดยไม่ต้องเทรนใหม่

In [ ]:
import joblib

MODEL_PATH = "crop_recommender.joblib"
joblib.dump({"model": best_model, "label_encoder": label_encoder, "features": features}, MODEL_PATH)
print(f"เซฟโมเดลแล้ว: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024**2:.1f} MB)")

# ดาวน์โหลดไฟล์ลงเครื่อง (ทำงานเฉพาะบน Google Colab)
try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    print("ไม่ได้รันบน Colab — ไฟล์อยู่ในโฟลเดอร์เดียวกับ notebook")

In [ ]:
# ทดสอบโหลดโมเดลกลับมาใช้ (ไม่ต้องเทรนใหม่) แล้วต้องทายได้เหมือนเดิม
saved = joblib.load(MODEL_PATH)
loaded_pred = saved["model"].predict(X_test)
print("โหลดโมเดลสำเร็จ — ทายตรงกับโมเดลเดิมทุกแถว:", bool((loaded_pred == y_pred_best).all()))

sample = pd.DataFrame([[90, 42, 43, 20.9, 82.0, 6.5, 202.9]], columns=saved["features"])
print("ตัวอย่างแปลง A ->", saved["label_encoder"].inverse_transform(saved["model"].predict(sample))[0])

## **Summary**

### **1. สรุปการทำงานของโมเดล**
*(ตัวเลขด้านล่างมาจากการรันด้วย `random_state=42` — ถ้ารันซ้ำจะได้ค่าเดิม · F1 = macro-F1)*

| Model | CV F1 (mean ± std) | Train F1 | Test Accuracy | Test F1 | ผิด (จาก 440) |
|---|---|---|---|---|---|
| Baseline (uniform – สุ่มทาย) | 0.046 ± 0.008 | 0.049 | 0.052 | 0.051 | 417 |
| Baseline (most_frequent) | 0.004 ± 0.000 | 0.004 | 0.045 | 0.004 | 420 |
| Logistic Regression (`C=100`) | 0.978 ± 0.006 | 0.990 | 0.984 | 0.984 | 7 |
| **Gaussian Naive Bayes (`var_smoothing=1e-9`)** | **0.995 ± 0.004** | **0.995** | **0.995** | **0.995** | **2** |
| Decision Tree (`gini, max_depth=None, min_samples_leaf=2`) | 0.986 ± 0.005 | 0.997 | 0.980 | 0.979 | 9 |

- **Baseline** สุ่มทายได้ accuracy เพียง 5.2% ส่วน most_frequent ได้ accuracy 4.5% แต่ F1 แค่ 0.004 → ตัวอย่าง **accuracy paradox** จึงต้องดู F1 ควบคู่เสมอ
- **ทุกโมเดลดีกว่า baseline เกือบ 20 เท่า** และค่า CV กับ test ใกล้เคียงกัน → ผลเชื่อถือได้
- **อ่าน bias–variance จาก Train F1 เทียบกับ CV F1:** Decision Tree train 0.997 แต่ CV 0.986 (ต้นลึก 17 ชั้น) → เริ่ม overfit · Naive Bayes train = CV = 0.995 → ไม่ overfit · Logistic Regression ต้องลด regularization (C สูง) ถึงจะดี (C=1 ได้ 0.968)
- **เลือก Gaussian Naive Bayes** เพราะ (1) CV Macro-F1 สูงสุดและ std ต่ำสุด (2) เป็นโมเดลที่ง่ายที่สุด เทรนเร็ว (3) ให้ความน่าจะเป็นที่ใช้แนะนำแบบ top-3 ได้ (4) สมมติฐาน "features อิสระกันภายในคลาส" เป็นจริงกับข้อมูลนี้ (P–K ภายในพืชเดียวกัน r ≈ 0)
  - *ทดลองเพิ่มนอกเนื้อหาในห้อง:* Random Forest ได้ CV 0.994 และ test 0.995 เท่ากัน → เมื่อผลเท่ากันจึงเลือกโมเดลที่ง่ายกว่า
- **จุดที่ยังผิด:** บน test 440 แถว Naive Bayes ผิดเพียง 2 แถว (rice → jute ทั้งคู่)
  - **rice ↔ jute** ต้องการ **ความชื้นสูง ฝนชุก N สูง** คล้ายกันมาก (Logistic Regression ก็พลาดคู่นี้ 3 ครั้ง)
  - **blackgram / lentil / mothbeans** (พืชตระกูลถั่ว) มีช่วงค่าดินทับซ้อนกัน — Decision Tree สับสนกลุ่มนี้มากที่สุด แต่ Naive Bayes ไม่พลาดเลย
- **Feature สำคัญที่สุด (Permutation Importance ของ Naive Bayes):** humidity > K > rainfall > P > N ส่วน temperature และ pH มีผลน้อยที่สุด

### **2. ตอบปัญหาที่ตั้งไว้**
> *ปัญหา: เกษตรกรเลือกพืชจากความเคยชิน ไม่ได้อิงค่าดิน/อากาศจริง*

- ระบบสามารถ **แนะนำพืชที่เหมาะกับค่าดิน + อากาศของแปลงได้ถูกต้อง 99.5%** (438 จาก 440 แปลง) บนข้อมูลที่โมเดลไม่เคยเห็น เทียบกับ 5.2% ถ้าสุ่มเดา
- ฟังก์ชัน `recommend_crops()` ให้ **พืช 3 อันดับพร้อมความมั่นใจ** → เกษตรกร/เจ้าหน้าที่ส่งเสริมการเกษตรระดับตำบล ใช้ประกอบการตัดสินใจ **ก่อนเริ่มฤดูปลูก** ร่วมกับปัจจัยอื่น เช่น ราคาตลาด
- ผลจาก feature importance บอกว่า **ความชื้น, โพแทสเซียม (K) และปริมาณฝน** คือสิ่งที่ควรวัด/ตรวจให้แม่นที่สุด → ช่วยให้รู้ว่าควรลงทุนตรวจดินค่าไหนก่อน

### **3. ข้อจำกัด — ทำไมยังไม่ควรนำไปใช้กับแปลงจริงทันที**
ความแม่นยำ 99.5% สะท้อนว่า **ชุดข้อมูลนี้แยกพืชได้ง่าย** ไม่ได้แปลว่าคำแนะนำจะถูกต้องในไร่จริง:
- **ข้อมูลน่าจะไม่ได้เก็บจากแปลงจริง** — ทุกพืชมี 100 แถวพอดี ค่าในแต่ละพืชกระจายในช่วงแคบอย่างสม่ำเสมอ ไม่พบ outlier ภายในพืชเลย และไม่ระบุแหล่งที่มา ภูมิภาค หรือปี (น่าจะเป็นข้อมูลอินเดีย)
- **โมเดลบอกได้แค่ว่าสภาพแปลง "คล้าย" พืชอะไร** ไม่ได้บอกว่าปลูกแล้วจะได้ผลผลิตดีหรือได้กำไร เพราะข้อมูลไม่มีผลผลิต ราคา หรือต้นทุน
- **โมเดลต้องเลือก 1 ใน 22 พืชเสมอ** แม้ค่าดินที่กรอกจะไม่เหมาะกับพืชชนิดใดเลย
- **ไม่ตรงบริบทไทย** — ไม่มีพืชหลักของไทย (อ้อย มันสำปะหลัง ยางพารา) ไม่มีชนิดดิน/แหล่งน้ำ/ชลประทาน และไม่ระบุหน่วยของ N, P, K และ rainfall ชัดเจน จึงเอาค่าจากใบตรวจดินจริงมากรอกตรงๆ ไม่ได้

**สิ่งที่ใช้ต่อได้:** ขั้นตอนทั้งหมด (Pipeline, CV, GridSearch, การประเมินผล, การเซฟโมเดล) นำไปใช้ซ้ำกับข้อมูลจริงของไทยได้ทันที → ตอนนี้ระบบอยู่ในระดับ **ต้นแบบ (proof of concept)** ใช้ช่วยคัดตัวเลือกให้เจ้าหน้าที่ ไม่ใช่ตัดสินแทนคน

### **4. แนวทางปรับปรุง**
1. **เก็บ/หาข้อมูลจริงของไทย** เช่น ผลตรวจดินจากกรมพัฒนาที่ดิน + ข้อมูลอากาศจากกรมอุตุนิยมวิทยา แล้วทดสอบว่าโมเดลยังแม่นอยู่หรือไม่ *(AI แนะนำ)*
2. **เพิ่ม feature ที่ข้อมูลชุดนี้ไม่มี** เช่น ชนิดดิน (soil type), ระดับน้ำใต้ดิน, ฤดูกาล, พื้นที่ชลประทาน *(AI แนะนำ)*
3. **เพิ่มมิติด้านเศรษฐกิจ** — แนะนำพืชโดยคำนึงถึงราคาขายและต้นทุนปุ๋ยด้วย เพื่อตอบปัญหารายได้เกษตรกรโดยตรง *(AI แนะนำ)*
4. **ลดจำนวน feature ที่ต้องวัด** — ทดลองตัด temperature / pH ที่ importance ต่ำสุด แล้วดูว่าความแม่นยำลดลงแค่ไหน เพื่อลดต้นทุนการตรวจดินของเกษตรกร *(AI แนะนำ)*
5. **ทำเป็นแอป/LINE bot** ที่ให้เจ้าหน้าที่ส่งเสริมการเกษตรกรอกค่าผลตรวจดินแล้วได้คำแนะนำทันที *(AI แนะนำ)*